In [2]:


import os
import sys
import zipfile
import tempfile
import urllib.request
from pathlib import Path

import pandas as pd

PERSON_ZIP_URL  = "https://www2.census.gov/programs-surveys/acs/data/pums/2023/1-Year/csv_pca.zip"
HOUSING_ZIP_URL = "https://www2.census.gov/programs-surveys/acs/data/pums/2023/1-Year/csv_hca.zip"

# Inside those ZIPs (CA-only):
PERSON_CSV_NAME  = "psam_p06.csv"
HOUSING_CSV_NAME = "psam_h06.csv"

OUTPUT_CSV = "california_pums_2023_complete.csv"

def download(url: str, dest: Path):
    """Download a URL to dest with a simple progress indicator."""
    def _reporthook(blocks, block_size, total_size):
        if total_size <= 0:
            return
        downloaded = blocks * block_size
        pct = downloaded / total_size * 100
        pct = 100 if pct > 100 else pct
        print(f"\r↓ {url}  [{pct:5.1f}%]", end="", flush=True)

    print(f"Downloading: {url}")
    urllib.request.urlretrieve(url, dest, _reporthook)
    print("\rDownloaded: ", dest.name, " " * 20, sep="")

def extract_member(zip_path: Path, member_name: str, out_dir: Path) -> Path:
    with zipfile.ZipFile(zip_path, "r") as zf:
        names = zf.namelist()
        if member_name not in names:
            raise FileNotFoundError(
                f"{member_name} not found in {zip_path.name}. "
                f"Available: {names[:5]}{'...' if len(names)>5 else ''}"
            )
        print(f"Extracting {member_name} from {zip_path.name}")
        zf.extract(member_name, path=out_dir)
        return out_dir / member_name

def main():
    workdir = Path.cwd()
    out_csv = workdir / OUTPUT_CSV
    if out_csv.exists():
        print(f"✅ {OUTPUT_CSV} already exists. Nothing to do.")
        return

    with tempfile.TemporaryDirectory() as td:
        tdir = Path(td)

        # 1) Download ZIPs
        person_zip  = tdir / "csv_pca.zip"
        housing_zip = tdir / "csv_hca.zip"
        download(PERSON_ZIP_URL, person_zip)
        download(HOUSING_ZIP_URL, housing_zip)

        # 2) Extract the CA CSVs
        person_csv_path  = extract_member(person_zip,  PERSON_CSV_NAME,  tdir)
        housing_csv_path = extract_member(housing_zip, HOUSING_CSV_NAME, tdir)

        # 3) Load with sensible dtypes to keep memory reasonable
        print("Loading person file…")
        persons = pd.read_csv(person_csv_path, low_memory=False)
        print(f"  persons: {len(persons):,} rows")

        print("Loading housing file…")
        housing = pd.read_csv(housing_csv_path, low_memory=False)
        print(f"  housing: {len(housing):,} rows")

        # 4) Merge on SERIALNO (one-to-many: each housing unit has many persons)
        if "SERIALNO" not in persons.columns or "SERIALNO" not in housing.columns:
            raise KeyError("SERIALNO key not found in input files.")

        print("Merging on SERIALNO …")
        merged = persons.merge(housing, on="SERIALNO", suffixes=("_P", "_H"))

        # Optional: sanity check for columns your framework needs
        required = ['AGEP','SEX','RAC1P','SCHL','WKHP','PINCP','PWGTP','ST','PUMA']
        missing_req = [c for c in required if c not in merged.columns]
        if missing_req:
            print(f"⚠️  Warning: missing expected columns: {missing_req}")

        # 5) Save in your expected filename
        merged.to_csv(out_csv, index=False)
        print(f"✅ Saved {out_csv.name} with {len(merged):,} rows")

        # Tiny preview
        print("Preview columns:", list(merged.columns)[:12], "…")

if __name__ == "__main__":
    try:
        main()
    except Exception as e:
        print("❌ Error:", e)
        sys.exit(1)


Downloading: https://www2.census.gov/programs-surveys/acs/data/pums/2023/1-Year/csv_pca.zip
Downloaded: w2.census.gov/programs-surveys/acs/data/pums/2023/1-Year/csv_pca.zip  [100.0%]csv_pca.zip                    
Downloading: https://www2.census.gov/programs-surveys/acs/data/pums/2023/1-Year/csv_hca.zip
Downloaded: w2.census.gov/programs-surveys/acs/data/pums/2023/1-Year/csv_hca.zip  [100.0%]csv_hca.zip                    
Extracting psam_p06.csv from csv_pca.zip
Extracting psam_h06.csv from csv_hca.zip
Loading person file…
  persons: 392,318 rows
Loading housing file…
  housing: 167,075 rows
Merging on SERIALNO …
⚠️  Warning: missing expected columns: ['ST', 'PUMA']
✅ Saved california_pums_2023_complete.csv with 392,318 rows
Preview columns: ['RT_P', 'SERIALNO', 'DIVISION_P', 'SPORDER', 'PUMA_P', 'REGION_P', 'STATE_P', 'ADJINC_P', 'PWGTP', 'AGEP', 'CIT', 'CITWP'] …
